In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import nltk
import os
import sys
import time
import math
import random
import argparse
from pathlib import Path
from typing import List
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Guarded import for bert-score
try:
    from bert_score import score as bertscore_score
    _HAS_BERTSCORE = True
except Exception:
    bertscore_score = None
    _HAS_BERTSCORE = False
    print("[warning] bert_score not installed; BERTScore will be skipped or return 0.0")

# Ensure punkt tokenizer is available
nltk.download("punkt", quiet=True)

# Configuration
DEFAULT_data_directory = "/content/drive/MyDrive/deshdeepak_nlu_dataset"
DEFAULT_OUT_DIR = os.path.join(DEFAULT_data_directory, "outcomecheck")
DSP_PREFIX = "spm_joint"
DVOCAB_SIZE = 12000
MAXIMUM_LENGTH = 128
SIZE_OF_BATCH = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Global pad id (will be set after loading SentencePiece)
PAD_ID = 1


def csv_reader(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path, dtype=str).fillna("")
    return None


def text_cleaning(text: str) -> str:
    """Cleaning Text: remove triple quotes, collapse repeated 'be able to', normalize whitespace."""
    if text is None:
        return ""
    text = str(text)
    text = text.replace('"""', '"').replace("''", "'")
    text = sub_repetable(text)
    text = " ".join(text.split()).strip()
    if text.startswith('"') and text.endswith('"'):
        text = text[1:-1]
    return text


def sub_repetable(text: str) -> str:
    import re
    return re.sub(r'\b(be able to\s+){2,}', 'be able to ', text)


# -------------------------
# Tokenizer (SentencePiece)
# -------------------------
def create_tokenizers(data_directory: Path, DSP_PREFIX: str = DSP_PREFIX, vocab_size: int = DVOCAB_SIZE) -> Path:
    model_file_path = data_directory / f"{DSP_PREFIX}.model"
    if model_file_path.exists():
        print(f"[tokenizer] Found existing SentencePiece model: {model_file_path}")
        return model_file_path

    print("------Creating tokenizer and combined text for SentencePiece training")
    combined_txt_data_direct = data_directory / "spm_input.txt"
    with open(combined_txt_data_direct, "w", encoding="utf-8") as fout:
        for file_name_str, col in [("train_sa_10000.csv", "Sentence_sa"), ("train_en_10000.csv", "Sentence_en"),
                                   ("dev_sa_1000.csv", "Sentence_sa"), ("dev_en_1000.csv", "Sentence_en")]:
            p = data_directory / file_name_str
            if p.exists():
                df = pd.read_csv(p, dtype=str).fillna("")
                for s in df[col].tolist():
                    fout.write(s.strip() + "\n")

    print("Now Tokenizer Training SentencePiece BPE model and it may take a minute")
    spm.SentencePieceTrainer.Train(
        f"--input={combined_txt_data_direct} --model_prefix={data_directory / DSP_PREFIX} --vocab_size={vocab_size} "
        f"--model_type=bpe --character_coverage=1.0 --unk_id=0 --pad_token_id=1 --bos_token_id=2 --eos_token_id=3"
    )
    model_file_path = data_directory / f"{DSP_PREFIX}.model"
    print(f"[tokenizer] Trained SentencePiece model saved to: {model_file_path}")
    return model_file_path


class NMTDataset(Dataset):
    def __init__(self, src_texts: List[str], tgt_texts: List[str], sp: spm.SentencePieceProcessor, MAXIMUM_LENGTH: int = MAXIMUM_LENGTH):
        assert len(src_texts) == len(tgt_texts)
        self.src = src_texts
        self.tgt = tgt_texts
        self.sp = sp
        self.MAXIMUM_LENGTH = MAXIMUM_LENGTH
        self.bos_token = sp.bos_id()
        self.eos_token = sp.eos_id()
        self.pad_token = sp.pad_id()

    def __len__(self):
        return len(self.src)

    def encode(self, text: str, add_bos_token_eos_token: bool = True) -> List[int]:
        ids = self.sp.EncodeAsIds(text)
        if add_bos_token_eos_token:
            ids = [self.bos_token] + ids[: self.MAXIMUM_LENGTH - 2] + [self.eos_token]
        else:
            ids = ids[: self.MAXIMUM_LENGTH]
        return ids

    def __getitem__(self, idx):
        src_ids = torch.tensor(self.encode(self.src[idx]), dtype=torch.long)
        tgt_ids = torch.tensor(self.encode(self.tgt[idx]), dtype=torch.long)
        return src_ids, tgt_ids


def combinng_sample(batch):
    """Collate function using global PAD_ID."""
    srcs, tgts = zip(*batch)
    srcs = pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgts = pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    return {"src": srcs, "tgt": tgts}


# Compact Transformer Model with fixed decode
class Transformertransform_positionalEncoding(nn.Module):
    def __init__(self, d_model: int, MAXIMUM_LENGTH: int = 5000):
        super().__init__()
        pe = torch.zeros(MAXIMUM_LENGTH, d_model)
        transform_position = torch.arange(0, MAXIMUM_LENGTH, dtype=torch.float).unsqueeze(1)
        trans_div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(transform_position * trans_div_term)
        pe[:, 1::2] = torch.cos(transform_position * trans_div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return x


class NMTModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, d_nhead: int = 8,
                 num_encoder_layers: int = 3, num_decoder_layers: int = 3,
                 dim_feedforward: int = 1024, dropout_rate: float = 0.1, pad_token_id: int = 1):
        super().__init__()
        self.pad_token_id = pad_token_id
        self.src_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.tgt_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.pos_enc = Transformertransform_positionalEncoding(d_model)
        self.pos_dec = Transformertransform_positionalEncoding(d_model)
        # Use batch_first=True for easier masking
        self.transformer = nn.Transformer(d_model, d_nhead, num_encoder_layers, num_decoder_layers,
                                          dim_feedforward, dropout_rate, batch_first=True)
        self.generator = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
        self._setup_parameter()

    def _setup_parameter(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src):
        src_key_padding_mask = (src == self.pad_token_id)
        src_emb = self.src_emb(src) * math.sqrt(self.d_model)
        src_emb = self.pos_enc(src_emb)
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        return memory, src_key_padding_mask

    def decode(self, tgt, memory, memory_key_padding_mask):
        """
        tgt: (B, T) token ids including bos_token and previous tokens
        memory: encoder memory
        memory_key_padding_mask: (B, S) boolean mask where True indicates pad_token
        """
        tgt_key_padding_mask = (tgt == self.pad_token_id)
        tgt_emb = self.tgt_emb(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.pos_dec(tgt_emb)
        T = tgt_emb.size(1)

        tgt_mask = self.transformer.generate_square_subsequent_mask(T).to(tgt.device)
        out = self.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return out

    def forward(self, src, tgt):
        memory, src_key_padding_mask = self.encode(src)
        dec_out = self.decode(tgt[:, :-1], memory, src_key_padding_mask)
        logits = self.generator(dec_out)  # (B, T-1, V)
        return logits


# Eval utils - Training
def loss_evaluation(logits: torch.Tensor, targets: torch.Tensor, pad_token_id: int = 1, label_smoothing: float = 0.1) -> torch.Tensor:
    V = logits.size(-1)
    targets_shift = targets[:, 1:].contiguous()
    loss_funct = nn.CrossEntropyLoss(ignore_index=pad_token_id, label_smoothing=label_smoothing)
    return loss_funct(logits.view(-1, V), targets_shift.view(-1))


def epoch_train(model: nn.Module, dataloader: DataLoader, optimizer, device: torch.device, scheduler=None):
    model.train()
    total_loss = 0.0
    steps = 0
    for batch in dataloader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)
        optimizer.zero_grad()
        logits = model(src, tgt)
        loss = loss_evaluation(logits, tgt, pad_token_id=model.pad_token_id)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(1, steps)


def single_decode(model: nn.Module, sp: spm.SentencePieceProcessor, src_ids: List[int], MAXIMUM_LENGTH: int = MAXIMUM_LENGTH, device: torch.device = DEVICE) -> str:
    model.eval()
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        memory, de_src_mask = model.encode(src)
        ys = torch.tensor([[sp.bos_id()]], dtype=torch.long).to(device)
        for _ in range(MAXIMUM_LENGTH - 1):
            out = model.decode(ys, memory, de_src_mask)
            logits = model.generator(out[:, -1, :])
            next_id = logits.argmax(dim=-1).unsqueeze(1)
            ys = torch.cat([ys, next_id], dim=1)
            if next_id.item() == sp.eos_id():
                break
    ids = ys.squeeze(0).tolist()
    ids = [i for i in ids if i not in (sp.bos_id(), sp.eos_id(), sp.pad_id())]
    return sp.DecodeIds(ids)


def decode_single_beam(model: nn.Module, sp: spm.SentencePieceProcessor, src_ids: List[int],
                       beam_size: int = 5, MAXIMUM_LENGTH: int = MAXIMUM_LENGTH, length_penalty: float = 0.6, device: torch.device = DEVICE) -> str:
    model.eval()
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        memory, de_src_mask = model.encode(src)
    beams = [(0.0, [sp.bos_id()], False)]
    completed = []
    for _ in range(MAXIMUM_LENGTH - 1):
        new_beams = []
        for score, seq, finished in beams:
            if finished:
                new_beams.append((score, seq, True))
                continue
            tgt = torch.tensor(seq, dtype=torch.long).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model.decode(tgt, memory, de_src_mask)
                logits = model.generator(out[:, -1, :])
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0).cpu().numpy()
            topk = np.argsort(-log_probs)[:beam_size]
            for idx in topk:
                nscore = score + float(log_probs[idx])
                nseq = seq + [int(idx)]
                finished_flag = (idx == sp.eos_id())
                new_beams.append((nscore, nseq, finished_flag))
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
        for b in beams:
            if b[2]:
                completed.append(b)
        if len(completed) >= beam_size:
            break
    if completed:
        scored = [(s / (len(seq) ** length_penalty), seq) for s, seq, _ in completed]
        best = max(scored, key=lambda x: x[0])[1]
    else:
        best = beams[0][1]
    best_ids = [i for i in best if i not in (sp.bos_id(), sp.eos_id(), sp.pad_id())]
    return sp.DecodeIds(best_ids)


def matrix_bleu(preds_list: List[str], refs: List[str]) -> float:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    refs_tokenized = [[r.split()] for r in refs]
    hyps_tokenized = [p.split() for p in preds_list]
    return corpus_bleu(refs_tokenized, hyps_tokenized, smoothing_function=SmoothingFunction().method1)


def matrix_bestscore(preds_list: List[str], refs: List[str], device: torch.device = DEVICE, batch_size: int = 64) -> float:
    """
    Compute BERTScore F1 (rescaled). Returns 0.0 if bert-score is not installed
    or if inputs are empty.
    """
    if not preds_list or not refs:
        return 0.0
    if not _HAS_BERTSCORE:
        return 0.0

    P, R, F1 = bertscore_score(
        preds_list,
        refs,
        lang="en",
        rescale_with_baseline=True,
        device=str(device) if isinstance(device, torch.device) else device,
        batch_size=batch_size
    )
    return float(F1.mean().item())


def main(args):
    global PAD_ID
    data_directory = Path(args.data_directory)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_sa_p = data_directory / "train_sa_10000.csv"
    train_en_p = data_directory / "train_en_10000.csv"
    dev_sa_p = data_directory / "dev_sa_1000.csv"
    dev_en_p = data_directory / "dev_en_1000.csv"
    test_sa_p = data_directory / "test_sa_1000.csv"
    test_en_p = data_directory / "test_en_1000.csv"

    print("Now mail pipline Checking dataset files in:", data_directory)
    for p in [train_sa_p, train_en_p, dev_sa_p, dev_en_p, test_sa_p]:
        print(f"  {p.name}: {'FOUND' if p.exists() else 'MISSING'}")

    train_df = csv_reader(train_sa_p)
    train_en_df = csv_reader(train_en_p)
    dev_df = csv_reader(dev_sa_p)
    dev_en_df = csv_reader(dev_en_p)
    test_sa_df = csv_reader(test_sa_p)
    test_en_df = csv_reader(test_en_p)

    if train_df is None or train_en_df is None:
        print("Main function ERROR: missing train files. Ensure train_sa_10000.csv and train_en_10000.csv are in data_directory.")
        sys.exit(1)
    train = train_df.merge(train_en_df, on="Source_id", how="inner", suffixes=("_sa", "_en"))
    print(f"[main] Train pairs: {len(train)}")
    dev = None
    if dev_df is not None and dev_en_df is not None:
        dev = dev_df.merge(dev_en_df, on="Source_id", how="inner", suffixes=("_sa", "_en"))
        print(f"[main] Dev pairs: {len(dev)}")
    if test_sa_df is not None:
        print(f"[main] Test (Sanskrit) rows: {len(test_sa_df)}")

    sp_model_file_path = data_directory / f"{DSP_PREFIX}.model"
    if not sp_model_file_path.exists():
        sp_model_file_path = create_tokenizers(data_directory, DSP_PREFIX=DSP_PREFIX, vocab_size=args.vocab_size)
    sp = spm.SentencePieceProcessor()
    sp.Load(str(sp_model_file_path))
    print("[main] Loaded SentencePiece model. Vocab size:", sp.GetPieceSize())

    # set global PAD_ID to actual SentencePiece pad id
    PAD_ID = sp.pad_id()

    vocab_size = sp.GetPieceSize()
    model = NMTModel(vocab_size, d_model=args.d_model, d_nhead=args.d_nhead,
                     num_encoder_layers=args.enc_layers, num_decoder_layers=args.dec_layers,
                     dim_feedforward=args.dim_feed, pad_token_id=sp.pad_id()).to(DEVICE)
    print("[main] Model parameter count:", sum(p.numel() for p in model.parameters()))

    ckpt_path = out_dir / args.checkpoint_name
    if args.mode == "infer" or (args.mode == "train" and ckpt_path.exists() and args.load_checkpoint):
        if ckpt_path.exists():
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
            model.to(DEVICE)
            print(f"[main] Loaded checkpoint from {ckpt_path}")
        else:
            print(f"[main] Checkpoint {ckpt_path} not found; continuing without loading.")

    if args.mode == "train":
        train_dataset = NMTDataset(train["Sentence_sa"].tolist(), train["Sentence_en"].tolist(), sp, MAXIMUM_LENGTH=MAXIMUM_LENGTH)
        train_loader = DataLoader(train_dataset, batch_size=args.SIZE_OF_BATCH, shuffle=True, collate_fn=combinng_sample)

        dev_dataset = None
        if dev is not None:
            dev_dataset = NMTDataset(dev["Sentence_sa"].tolist(), dev["Sentence_en"].tolist(), sp, MAXIMUM_LENGTH=MAXIMUM_LENGTH)

        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.0)

        if args.use_warmup:
            def lr_lambda(step):
                warmup = max(1, args.warmup_steps)
                step = max(1, step)
                return (model.d_model ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)
            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        else:
            scheduler = None

        best_dev_bleu = -1.0
        no_improve = 0
        for epoch in range(1, args.epochs + 1):
            t0 = time.time()
            train_loss = epoch_train(model, train_loader, optimizer, DEVICE, scheduler)
            t1 = time.time()
            print(f"[train] Epoch {epoch} | Train loss {train_loss:.4f} | Time {t1 - t0:.1f}s")

            if dev_dataset is not None:
                preds_list, refs = [], []
                eval_n = min(len(dev_dataset), args.dev_eval_n)
                for i in range(eval_n):
                    src_ids = dev_dataset.encode(dev_dataset.src[i])
                    if args.eval_beam == 1:
                        pred = single_decode(model, sp, src_ids, MAXIMUM_LENGTH=MAXIMUM_LENGTH, device=DEVICE)
                    else:
                        pred = decode_single_beam(model, sp, src_ids, beam_size=args.eval_beam, MAXIMUM_LENGTH=MAXIMUM_LENGTH, device=DEVICE)
                    preds_list.append(pred)
                    refs.append(dev_dataset.tgt[i])
                dev_bleu = matrix_bleu(preds_list, refs)
                print(f"[train] Dev NLTK BLEU (sample input {eval_n}): {dev_bleu:.6f}")

                if dev_bleu > best_dev_bleu:
                    best_dev_bleu = dev_bleu
                    torch.save(model.state_dict(), ckpt_path)
                    print(f"[train] New best dev BLEU. Saved checkpoint to {ckpt_path}")
                    no_improve = 0
                else:
                    no_improve += 1
                    print(f"[train] No improvement count: {no_improve}/{args.patience}")
                    if no_improve >= args.patience:
                        print("[train] Early stopping triggered.")
                        break
            else:
                torch.save(model.state_dict(), ckpt_path)
                print(f"[train] Saved checkpoint to {ckpt_path}")

        print("[train] Training finished......")

    if args.mode == "infer":
        if test_sa_df is None:
            print("[infer] test_sa_1000.csv not found in data_directory. Place test_sa_1000.csv and rerun.")
            return

        if ckpt_path.exists():
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
            model.to(DEVICE)
            print(f"[infer] Loaded checkpoint from {ckpt_path} for inference.")
        else:
            print("[infer] No checkpoint found; using current model weights (may be untrained).")

        source_ids = test_sa_df["Source_id"].tolist()
        sentences = test_sa_df["Sentence_sa"].tolist()
        preds_list = []

        print("Starting inference on test set...")
        start = time.perf_counter()
        for s in sentences:
            src_ids = [sp.bos_id()] + sp.EncodeAsIds(s)[: MAXIMUM_LENGTH - 2] + [sp.eos_id()]
            if args.beam == 1:
                pred = single_decode(model, sp, src_ids, MAXIMUM_LENGTH=MAXIMUM_LENGTH, device=DEVICE)
            else:
                pred = decode_single_beam(model, sp, src_ids, beam_size=args.beam, MAXIMUM_LENGTH=MAXIMUM_LENGTH, device=DEVICE)
            preds_list.append(text_cleaning(pred))
        end = time.perf_counter()
        total_time = end - start
        print(f"Inference time: {total_time:.2f}s for {len(sentences)} sentences ({total_time/len(sentences):.4f}s/sentence)")

        if test_en_df is not None:
            refs = test_en_df["Sentence_en"].tolist()
            nltk_bleu = matrix_bleu(preds_list, refs)
            bert_f1 = matrix_bestscore(preds_list, refs, device=DEVICE, batch_size=args.bert_SIZE_OF_BATCH)
            print(f" Test NLTK BLEU: {nltk_bleu:.6f}")
            print(f" Test BERTScore F1 (rescaled): {bert_f1:.4f}")
        else:
            print("test_en_1000.csv not found; skipping metric computation.")
        submission = pd.DataFrame({"Source_id": source_ids, "Sentence_en": preds_list})
        submission.loc[submission["Sentence_en"].astype(str).str.strip() == "", "Sentence_en"] = "No translation available"
        submission_path = data_directory / "submission.csv"
        submission.to_csv(submission_path, index=False, encoding="utf-8")
        print(f"Saved submission.csv to {submission_path}")

    total_params = sum(p.numel() for p in model.parameters())
    print(f"[main] Total model parameters: {total_params}")


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description="Sanskrit->English NMT assignment script (fixed)")
    parser.add_argument("--mode", type=str, choices=["train", "infer"], default="train")
    parser.add_argument("--data_directory", type=str, default=DEFAULT_data_directory)
    parser.add_argument("--out_dir", type=str, default=DEFAULT_OUT_DIR)
    parser.add_argument("--checkpoint_name", type=str, default="perfect_model.pt")
    parser.add_argument("--load_checkpoint", action="store_true", help="Load checkpoint at start if present")
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--SIZE_OF_BATCH", type=int, default=SIZE_OF_BATCH)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--vocab_size", type=int, default=DVOCAB_SIZE)
    parser.add_argument("--d_model", type=int, default=256)
    parser.add_argument("--d_nhead", type=int, default=8)
    parser.add_argument("--enc_layers", type=int, default=3)
    parser.add_argument("--dec_layers", type=int, default=3)
    parser.add_argument("--dim_feed", type=int, default=1024)
    parser.add_argument("--beam", type=int, default=5, help="Beam size for inference (1 = greedy)")
    parser.add_argument("--eval_beam", type=int, default=1, help="Beam size used during dev evaluation")
    parser.add_argument("--dev_eval_n", type=int, default=200, help="Number of dev examples to evaluate per epoch")
    parser.add_argument("--patience", type=int, default=5, help="Early stopping patience on dev BLEU")
    parser.add_argument("--use_warmup", action="store_true", help="Use Transformer warmup LR schedule")
    parser.add_argument("--warmup_steps", type=int, default=4000, help="Warmup steps for LR schedule")
    parser.add_argument("--bert_SIZE_OF_BATCH", type=int, default=64, help="Batch size for BERTScore computation")
    return parser.parse_args(argv)


if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        args = parse_args([])
    else:
        args = parse_args()
    main(args)
